# Step 1 — OpenVLA Real Profiling (HuggingFace)

Downloads **`openvla/openvla-7b`** and measures **real** inference latency. Results overwrite `research/results/step1_profiling/` (back up mock outputs first if you want to keep them).

## Before you run

| Requirement | Minimum | Recommended |
|-------------|---------|-------------|
| Machine | Linux + **NVIDIA GPU** | Lab workstation / HPC node |
| VRAM | 7 GB with INT4 | 13+ GB (FP16) |
| Disk | ~20 GB free | HF cache + weights |
| CUDA | 12.1+ | Matches PyTorch build |

**Will not work on Mac alone** — this script uses CUDA (`.cuda()`), not MPS/CPU.

### One-time setup (on GPU machine)

```bash
cd research_summer_2026/research
conda create -n openvla_g1 python=3.10 -y && conda activate openvla_g1
pip install -e .
pip install transformers accelerate timm sentencepiece einops huggingface_hub
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# If VRAM < 13 GB:
pip install bitsandbytes
```

Optional: `huggingface-cli login` if download fails (model is usually public).

**CLI equivalent:**
```bash
python -m src.step1_profile_openvla --n_trials 100 --no-mock-fallback \
  --instruction "Pick up the pen on the book on the table and give it to me"
```

In [1]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    RESEARCH_DIR = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "src" / "step1_profile_openvla.py").is_file():
    RESEARCH_DIR = NOTEBOOK_DIR
else:
    RESEARCH_DIR = NOTEBOOK_DIR / "research_summer_2026" / "research"
    if not RESEARCH_DIR.is_dir():
        RESEARCH_DIR = NOTEBOOK_DIR / "research"

RESEARCH_DIR = RESEARCH_DIR.resolve()
assert (RESEARCH_DIR / "src").is_dir(), f"src package not found under: {RESEARCH_DIR}"

os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step1_profiling'}")

Research root : /Users/andrew/Desktop/competitions/research_summer_2026/research
Results go to : /Users/andrew/Desktop/competitions/research_summer_2026/research/results/step1_profiling


## Pre-flight checks (run before downloading 13 GB)

In [2]:
import shutil
import torch

checks_ok = True

if not torch.cuda.is_available():
    checks_ok = False
    print("FAIL: CUDA not available. Use a machine with an NVIDIA GPU.")
else:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"OK  : GPU {props.name}")
    print(f"OK  : VRAM {vram_gb:.1f} GB")
    if vram_gb < 12:
        print("WARN: VRAM < 12 GB — set USE_INT4 = True in the next cell")

for pkg in ("transformers", "timm", "PIL"):
    try:
        __import__(pkg if pkg != "PIL" else "PIL")
        print(f"OK  : {pkg}")
    except ImportError:
        checks_ok = False
        print(f"FAIL: pip install {pkg}")

disk_gb = shutil.disk_usage(RESEARCH_DIR).free / (1024 ** 3)
print(f"OK  : {disk_gb:.0f} GB free disk under research/")
if disk_gb < 20:
    print("WARN: < 20 GB free — model cache may need more space")

assert checks_ok, "Fix failed checks before continuing."


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/Users/andrew/Downloads/RESEARCH-01/workshop/ai-deeds/challenge/Smart Buildings/.venv311/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/andrew/Downl

FAIL: CUDA not available. Use a machine with an NVIDIA GPU.
FAIL: pip install transformers
FAIL: pip install timm
OK  : PIL
OK  : 19 GB free disk under research/
WARN: < 20 GB free — model cache may need more space


AssertionError: Fix failed checks before continuing.

In [ ]:
# Experiment config — REAL OpenVLA (edit here)
MOCK = False
N_TRIALS = 100          # paper-style: 100; use 10 for a quick smoke test
INSTRUCTION = "Pick up the pen on the book on the table and give it to me"
MODEL_ID = "openvla/openvla-7b"
USE_INT4 = False        # True if VRAM < 13 GB (requires bitsandbytes)

# Optional: copy mock results before overwrite
BACKUP_MOCK_RESULTS = False
if BACKUP_MOCK_RESULTS:
    import shutil
    src = RESEARCH_DIR / "results" / "step1_profiling"
    dst = RESEARCH_DIR / "results" / "step1_profiling_mock_backup"
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Backed up to {dst}")

## Load model + run profiling

First run downloads ~13 GB from HuggingFace (can take several minutes).

In [ ]:
from src.step1_profile_openvla import (
    MockG1SimEnv,
    OpenVLAWrapper,
    G1_DOF,
    TARGET_HZ,
    RESULTS_DIR,
    profile_openvla,
    save_logs,
    plot_profiling_report,
)

env = MockG1SimEnv(image_size=(224, 224))
model = OpenVLAWrapper(
    model_id=MODEL_ID,
    use_int4=USE_INT4,
    action_dim=G1_DOF,
    allow_mock_fallback=False,  # fail loud — never silent mock on this notebook
)
assert model.model is not None, "Model failed to load but did not raise — check OpenVLAWrapper"

report, records = profile_openvla(
    model=model,
    env=env,
    n_trials=N_TRIALS,
    instruction=INSTRUCTION,
)

save_logs(report, records)
plot_profiling_report(report, records)

print("\n" + "=" * 60)
print("  REAL PROFILING COMPLETE — Week 1-2 Deliverable")
print("=" * 60)
print(f"  Model          : {report.model_id}")
print(f"  Instruction    : {INSTRUCTION}")
print(f"  Mean latency   : {report.mean_latency_ms:.1f} ± {report.std_latency_ms:.1f} ms")
print(f"  Control rate   : {report.mean_hz:.2f} Hz  (target: {TARGET_HZ} Hz)")
print(f"  Frequency gap  : {report.frequency_gap:.1f}x")
print(f"  Failure rate   : {report.failure_rate * 100:.1f}%")
print(f"  GPU memory     : {report.mean_gpu_mem_gb:.2f} GB")
print(f"  Outputs        : {RESULTS_DIR.resolve()}")
print("=" * 60)
print("Next: copy numbers into src/step1_observation_log.md")

## Inspect results

In [ ]:
import json
from IPython.display import Image, display

summary_path = RESULTS_DIR / "profiling_summary.txt"
report_path = RESULTS_DIR / "profiling_report.json"
fig_path = RESULTS_DIR / "openvla_profiling_report.png"

if summary_path.exists():
    print(summary_path.read_text())

if report_path.exists():
    r = json.loads(report_path.read_text())
    display({k: r[k] for k in [
        "model_id", "n_trials", "mean_latency_ms", "mean_hz",
        "frequency_gap", "failure_rate", "mean_gpu_mem_gb",
    ] if k in r})

if fig_path.exists():
    display(Image(filename=str(fig_path)))